# LAB 1: Making Propositional Logic (PL) and Querying PL Knowledge base
In this lab you will implement propositional logic which will be the foundation of our AI agent.
Initially it might seem strange to implement propositional logic, since python already has it implemented in the simple ```not```, ```and```, ```or```, and ```==``` operators (implication can be derived from these as well).
While its true that any propositional logic statement can be evaluated using Python, these statements cannot be stored in their unevaluated form (which is desirable to form a Knowledge Base- KB).
Additionally, we cannot create variables without assigning them value in standard Python, but when we later want to iterate over all possible models the variables will need to take on many possible values.

So we will need to implement our own version of propositional logic in Python.
For variables we will use strings to represent them and later we can use dictionaries to represent models and then look up the value of each variable in that model.
For True and False we will simply use the existing ```True``` and ```False``` values in Python.
For all the operators we will implement classes that contain the logic sentences they operate on.

Task 1:
 Given  a skeleton for the "and" operator, you need to expand it to each of the other operators.


Task 2: 
Add an option, which allows to deal with repeated use of the same operation over multiple sentences.

Task 3: 
Implement a function that given a binary operator and a list of sentences chains them togehter (e.g. ```fun(And, [s1, s2, s3]) -> And(S1, And(s2, s3))```).


In [11]:
# Task 1: classes for logical sentences

class And:
    def __init__(self, A, B):
        self.A = A
        self.B = B
        
    def __repr__(self):
        return f"({self.A} & {self.B})"


class Or:
    def __init__(self, A, B):
        self.A = A
        self.B = B

    def __repr__(self):
        return f"({self.A} | {self.B})"


class Equals:
    # Logical equivalence (iff): A <-> B
    def __init__(self, A, B):
        self.A = A
        self.B = B

    def __repr__(self):
        return f"({self.A} <-> {self.B})"


class Implies:
    # Logical implication: A -> B
    def __init__(self, A, B):
        self.A = A
        self.B = B

    def __repr__(self):
        return f"({self.A} -> {self.B})"


class Not:
    # Unary operator
    def __init__(self, A):
        self.A = A

    def __repr__(self):
        return f"~({self.A})"


# Task 2, 3: chaining function 

def chain(op, sentences):
    """
    Given a binary operator class (And, Or, Equals, Implies)
    and a list [s1, s2, s3, ...],
    build: op(s1, op(s2, op(s3, ...))).

    Example:
        chain(And, [s1, s2, s3]) -> And(s1, And(s2, s3))
    """
    if not sentences:
        raise ValueError("Need at least one sentence")
    if len(sentences) == 1:
        return sentences[0]

    # Right-associative: s1 op (s2 op (s3 op ...))
    result = sentences[-1]
    for i in range(len(sentences) - 2, -1, -1):
        result = op(sentences[i], result)
    return result


In [ ]:
# Example usage and testing
P, Q, R = "P", "Q", "R"

s1 = And(P, Q)
s2 = Or(Not(P), Q)
s3 = Implies(P, Q)
s4 = Equals(And(P, Q), R)

# Print the sentences
print(s1)  # (P & Q)
print(s2)  # ~(P) | Q, but printed as ~(P) inside Or: (~(P)| Q)
print(s3)  # (P -> Q)
print(s4)  # ((P & Q) <-> R)

# Testing the chaining function
big_and = chain(And, [P, Q, R])
big_or = chain(Or, [P, Q, R])
big_implies = chain(Implies, [P, Q, R])
big_equals = chain(Equals, [P, Q, R])

# Print the chained sentences
print(big_and)  # (P & (Q & R))
print(big_or)   # (P | (Q | R))
print(big_implies)  # (P -> (Q -> R))
print(big_equals)   # (P <-> (Q <-> R))


(P & Q)
(~(P) | Q)
(P -> Q)
((P & Q) <-> R)
(P & (Q & R))
(P | (Q | R))
(P -> (Q -> R))
(P <-> (Q <-> R))


## Evaluating propositional sentences in a given model

Being able to form sentences in propositional logic is not very useful if we cannot actually evaluate them. A start of the function can be found below, but it is missing most things. 

Task 4:
Implement a fuction ```evaluate``` that takes a sentence in propositional logic and a dictonary with the truth values of the different variables in a given model and returns the truth value of the sentence in that model.

Note that he function for replacing the variables with their correct truth-value will currently give an error if a variable is missing, but we want it to return ```None``` to represent an unknown value instead.

Task 5:
Add a new thing to the mix to make evaluation of large models more efficient down the line.
The thing we will add is the concept of an unknown or uninteresting truth value using the Python ```None```-type.
The idea with unknown values is that instead of iterating over every value for every variable to evaluate a model, we can decide to only iterate over some variables and set the others as unknown.
However if an important variable is set to unknown we might not be able to get the correct answer.
In order to protect against this the ```evaluate``` function will return ```True``` if the sentence is true, ```False``` if its false, and ```None``` if we cannot know since some variables are unknown to us.
In order to deal with the ```None``` values you will have to think what the answer to many different operators should be. For example, we know that ```And(False, None)``` should be ```False``` since ```And``` always returns ```False``` when at least one sub-sentence is ```False```.
You will also need to experiment with Python to see if the Python operators give the correct behavior. Does ```False == None``` return ```None```.
If not, how to deal with it.




In [28]:
# Task 4,5: evaluation function

def evaluate(sentence, variables):
    """
    Evaluate a PL sentence in a given model.
    Returns: True, False, or None (unknown).
    """

    # Literal values
    if sentence is True or sentence is False or sentence is None:
        return sentence

    # Variable (string)
    if isinstance(sentence, str):
        return variables.get(sentence, None)  # None if missing

    # AND
    if isinstance(sentence, And):
        a = evaluate(sentence.A, variables)
        b = evaluate(sentence.B, variables)
        if a is False or b is False:
            return False
        if a is True and b is True:
            return True
        return None

    # OR
    if isinstance(sentence, Or):
        a = evaluate(sentence.A, variables)
        b = evaluate(sentence.B, variables)
        if a is True or b is True:
            return True
        if a is False and b is False:
            return False
        return None

    # NOT
    if isinstance(sentence, Not):
        v = evaluate(sentence.A, variables)
        if v is True:
            return False
        if v is False:
            return True
        return None

    # IMPLIES: A -> B
    if isinstance(sentence, Implies):
        a = evaluate(sentence.A, variables)
        b = evaluate(sentence.B, variables)

        if a is True:
            return b            # whatever B is (True / False / None)
        if a is False:
            return True         # False -> anything is True
        # a is None
        if b is True:
            return True
        return None

    # EQUALS: A <-> B (iff)
    if isinstance(sentence, Equals):
        a = evaluate(sentence.A, variables)
        b = evaluate(sentence.B, variables)
        if a is None or b is None:
            return None
        return (a == b)

    raise TypeError(f"Cannot evaluate sentence of type {type(sentence)}")


### Testing the evaluate function

Task 6: We have now created a propositional logic grammar and a function to evaluate the truth value of sentences for a given model.
In order to make sure everything is correct we introduce a testing space below and ask you to implement and evaluate a number of sentences for a given model in order to see that you get the correct result.

Below you are given a model and are then supposed to implement and run evaluation on the sentences below, with the given expected results:

#### Example 10 (Group 46+):
- $temperature\_high$ is ```True```
- $ac\_on \rightarrow temperature\_high$ is ```True```
- $(\neg windows\_open \rightarrow \neg ac\_on) \leftrightarrow False$ is ```True```
- $(temperature\_high \wedge humidity\_high) \rightarrow maintenance\_mode$ is ```None```
- $windows\_open \rightarrow maintenance\_mode$ is ```True```
- $\neg temperature\_high \leftrightarrow maintenance\_mode$ is ```None```
- $\neg lights\_on$ is ```True```
- $(temperature\_high \wedge humidity\_high \wedge lights\_on) \leftrightarrow False$ is ```True```

Choose valid test model below depending on your assigned example

In [ ]:
# Task 6: Example 10
test_model = {
    'temperature_high': True,
    'ac_on': True,
    'windows_open': False,
    'humidity_high': True,
    'lights_on': False,
}

# -------Sentences to evaluate:---------

# temperature_high  → True
s1 = "temperature_high"

# ac_on → temperature_high  → True
s2 = Implies("ac_on", "temperature_high")

# (¬windows_open → ¬ac_on) ↔ False  → True
s3 = Equals(
    Implies(
        Not("windows_open"),
        Not("ac_on")
    ),
    False
)

# (temperature_high ∧ humidity_high) → maintenance_mode  → None
s4 = Implies(
    And("temperature_high", "humidity_high"),
    "maintenance_mode"
)

# windows_open → maintenance_mode  → True
s5 = Implies("windows_open", "maintenance_mode")

# ¬temperature_high ↔ maintenance_mode  → None
s6 = Equals(
    Not("temperature_high"),
    "maintenance_mode"
)

# ¬lights_on  → True
s7 = Not("lights_on")

# (temperature_high ∧ humidity_high ∧ lights_on) ↔ False  → True
s8 = Equals(
    chain(And, ["temperature_high", "humidity_high", "lights_on"]),
    False
)


# Print evaluation results
print("1:", evaluate(s1, test_model))
print("2:", evaluate(s2, test_model))
print("3:", evaluate(s3, test_model))
print("4:", evaluate(s4, test_model))
print("5:", evaluate(s5, test_model))
print("6:", evaluate(s6, test_model))
print("7:", evaluate(s7, test_model))
print("8:", evaluate(s8, test_model))


1: True
2: True
3: True
4: None
5: True
6: None
7: True
8: True


### Querying for possible models from a Knowledge Base

Task 7: Now we are ready to create a knowledge base.
Making a knowledge base is not that difficult, it's simply a collection of logical sentences that are known (or presumed) to be true in a given environment.

To create this collection we will use the Python ```set``` data structure.
The reason is that sets can only contain one of each element so we don't risk adding the same sentence multiple times.
Sets can be created by listing  a number of elements in curly brackets (```{}```), and elements can be added and removed with the ```.add()``` and ```.remove()``` functions.

The more difficult part is then creating a function for querying the knowledge base for whether a new sentence is true or not.
One simple way to do this query is to test every possible model (every possible combination of truth values for the variables).
For each model we figure out whether the query sentence is True or False, we then also check that all sentences in the knowledge base are True for that model.
Since we "know" that each sentence in the knowledge base should be true, any model that doesn't align with this can be discarded as an "invalid" model of our environment.
For all "valid" models we then return whether that model is True or False.

Of course many environments have a huge amount of variables which would then require us to check a huge number of models.
If we have $n$ variables in the environment we would need to check each sentence $n^2$ times.
However,  not every variable is necessarily relevant for our query sentence.

Using our knowledge of the environment we might be able to deduce the minimum number of variables that are relevant for our query and only test all combinations of them.
This is why we implemented the concept of unknown (```None```) values earlier.
Due to this we also consider a model "valid" if sentence is the knowledge base evaluate as unknown.
This comes with a risk of course, if we mistakenly exclude a relevant variable in our query a model that should be "invalid" might be considered "valid" since a False evaluation might now be unknown.
Additionally, we need to return all "valid" models that are unknown.

We can then use this query function to figure things about the environment.
If a query sentence only gives "valid" models in which it is True, then we know the sentence is True in the environment.
Conversely, if all returned models are False, the sentence is False in the environment.
If it returns some True models and some False models we know that, given what we know of the environment we cannot yet tell whether the sentence is True or not (in some advanced scenarios we might be able to use the number of True/False models to estimate the likelihood of the query being True/False in the actual environment).
If some of the returned models are unknown models, we know that there are relevant variables that were not considered in the query.
This likely means we have an error in the code that determines which variables should be relevant for a query.
Note that many variables that are not in the query sentence could still be relevant.

Finally, if the function returns no valid models at all, this means that there are contradictions in the knowledge base and something has gone wrong in its construction.

Below you will find the skeleton of the query function for you to complete.

In [ ]:
# Task 7: query function

import itertools

# Query function
def query(query_sentence, KB, variables):
    '''
    For a given query and knowledge base, returns the models where all KB sentences are true
    in three lists depending on if the query is True, False, or None
    '''
    vars_list = list(variables)

    true_models = []
    false_models = []
    none_models = []

    # All combinations of True/False for the relevant variables
    for values in itertools.product([False, True], repeat=len(vars_list)):
        model = dict(zip(vars_list, values))

        # Check if model is valid for the KB
        valid = True
        for sentence in KB:
            val = evaluate(sentence, model)
            if val is False:   # KB sentence cannot be False
                valid = False
                break

        if not valid:
            continue

        # Evaluate the query in this valid model
        q_val = evaluate(query_sentence, model)

        if q_val is True:
            true_models.append(model)
        elif q_val is False:
            false_models.append(model)
        else:
            none_models.append(model)

    return true_models, false_models, none_models


### Testing the querying function

No complex function is complete without testing.
To do this we will use the sentences from testing our evaluation function as the knowledge base.
You will be given a list of "relevant" variables and then you will query the knowledge base with the following sentences with the given expected results.

The query function should be designed after the design requirments stated (arg/return type). 

The solution is accepted only if the querying function exactly returns the stated expected result of each query.

#### Example 10 (Group 46-50+)
- $True$ returns 2 True models (and 0 False/None models).
- $temperature\_high \leftrightarrow \neg ac\_on$ returns 2 False models. 
- $lights\_on \wedge humidity\_high$ returns 2 False models.
- $ac\_on \leftrightarrow lights\_on$ returns 2 False models.
- $(ac\_on \leftrightarrow \neg humidity\_high) \leftrightarrow False$ returns 1 True model and 1 False model. 

Below you have test variables. Select test variables based on your assigned example. 

In [25]:
# Relevant variables for Example 10
test_variables = {"temperature_high", "ac_on", "windows_open", "humidity_high", "lights_on"}

# Sentences from Task 6 (Example 10)
test_model = {
    'temperature_high': True,
    'ac_on': True,
    'windows_open': False,
    'humidity_high': True,
    'lights_on': False,
}

# Sentences to form the knowledge base
s1 = "temperature_high"
s2 = Implies("ac_on", "temperature_high")
s3 = Equals(Implies(Not("windows_open"), Not("ac_on")), False)
s4 = Implies(And("temperature_high", "humidity_high"), "maintenance_mode")
s5 = Implies("windows_open", "maintenance_mode")
s6 = Equals(Not("temperature_high"), "maintenance_mode")
s7 = Not("lights_on")
s8 = Equals(chain(And, ["temperature_high", "humidity_high", "lights_on"]), False)

# Knowledge base: all sentences used to test evaluate
KB = {s1, s2, s3, s4, s5, s6, s7, s8}


In [ ]:
# Perform queries

# Query: True
q1 = True
t1, f1, n1 = query(q1, KB, test_variables)
print("Q1 True models:", len(t1), "False models:", len(f1), "None models:", len(n1))

# temperature_high ↔ ¬ac_on
q2 = Equals("temperature_high", Not("ac_on"))
t2, f2, n2 = query(q2, KB, test_variables)
print("Q2 True models:", len(t2), "False models:", len(f2), "None models:", len(n2))

# lights_on ∧ humidity_high
q3 = And("lights_on", "humidity_high")
t3, f3, n3 = query(q3, KB, test_variables)
print("Q3 True models:", len(t3), "False models:", len(f3), "None models:", len(n3))

# ac_on ↔ lights_on
q4 = Equals("ac_on", "lights_on")
t4, f4, n4 = query(q4, KB, test_variables)
print("Q4 True models:", len(t4), "False models:", len(f4), "None models:", len(n4))

# (ac_on ↔ ¬humidity_high) ↔ False
q5 = Equals(Equals("ac_on", Not("humidity_high")), False)
t5, f5, n5 = query(q5, KB, test_variables)
print("Q5 True models:", len(t5), "False models:", len(f5), "None models:", len(n5))


Q1 True models: 2 False models: 0 None models: 0
Q2 True models: 0 False models: 2 None models: 0
Q3 True models: 0 False models: 2 None models: 0
Q4 True models: 0 False models: 2 None models: 0
Q5 True models: 1 False models: 1 None models: 0


### Before submitting to Canvas

Important! Make sure you follow design requirements for the evaluate and query function (correct argument- and return type). The notebook will be unit tested and verified. 

